# Buncefield-Style Tutorial (Deep Safety)

This notebook demonstrates a Buncefield-style vapor cloud consequence workflow using Deep Safety's physics layer.

Flow: source term -> dispersion screening -> explosion screening -> constant sensitivity.

In [ ]:
from deepsafety.source_models import solve_source_model
from deepsafety.dispersion_service import solve_dispersion_model
from deepsafety.fire_explosion_models import solve_fire_explosion_model
from deepsafety.constants import get_constant_value

print('TNT heat of explosion constant (kJ/kg):', get_constant_value('shared.tnt_heat_of_explosion_kj_kg'))

In [ ]:
source_inputs = {
    'source_subtype': 'pipe',
    'duration_s': 120.0,
    'upstream_pressure_pa': 8_000_000.0,
    'downstream_pressure_pa': 101_325.0,
    'temperature_k': 293.15,
    'heat_capacity_ratio': 1.28,
    'molecular_weight_kg_kmol': 58.12,
    'hole_diameter_m': 0.01,
    'inventory_mass_kg': 2500.0,
}

source_result = solve_source_model('gas_release', source_inputs)
source_result

In [ ]:
dispersion_inputs = {
    'release_rate_kg_s': source_result['average_release_rate_kg_s'],
    'release_height_m': 1.0,
    'wind_speed_m_s': 3.0,
    'stability_class': 'D',
    'x_m': 400.0,
    'y_m': 0.0,
    'z_m': 1.5,
}

dispersion_result = solve_dispersion_model('gaussian_plume', dispersion_inputs)
dispersion_result

In [ ]:
explosion_inputs = {
    'cloud_mass_kg': source_result['total_mass_kg'] * 0.35,
    'heat_of_combustion_kj_kg': 46_000.0,
    'ignition_delay_s': 25.0,
    'congestion_factor': 1.2,
    'distance_m': 300.0,
}

vce_result = solve_fire_explosion_model('vce', explosion_inputs)
vce_result

## Constants sensitivity example

For API-based studies, you can override constants per request. Example:

```json
{
  "constants": {
    "fire.default_radiative_fraction": 0.40
  }
}
```